# Week 8 Lab：Consistency functions、CD 與 CT

## 學習目標
- 在有精確 flow map 的 toy 上驗證 consistency function 沿軌跡不變。
- 比較有限差分 consistency distillation（CD）與連續時間導數 residual（CT）。
- 量化一步近似與多步極限之間的差距。

> **誠實註記**：下方 flow、teacher 與 surrogate 都是明寫的解析函數；沒有真實 checkpoint，surrogate 誤差也是刻意注入。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

SEED = 808
rng = np.random.default_rng(SEED)
OMEGA, DECAY = 1.15, .32

def rotate(x, angle):
    c, s = np.cos(angle), np.sin(angle)
    R = np.array([[c, -s], [s, c]])
    return np.asarray(x) @ R.T

def flow_map(x, t, s):
    scale = np.exp(-DECAY * (s - t))
    angle = OMEGA * (s ** 2 - t ** 2)
    return scale * rotate(x, angle)

def f_exact(x, t):
    return flow_map(x, t, 0.0)

def f_surrogate(x, t, error=.14):
    x = np.asarray(x)
    wobble = error * t * (1 - t) * np.stack([np.sin(2 * x[..., 1]), np.cos(2 * x[..., 0])], axis=-1)
    return f_exact(x, t) + wobble

x_clean = rng.normal(size=(12, 2))
ts = np.linspace(0, 1, 120)
fig, ax = plt.subplots(figsize=(5.5, 5))
for x in x_clean:
    trajectory = np.array([flow_map(x, 0, t) for t in ts])
    ax.plot(trajectory[:, 0], trajectory[:, 1], alpha=.7)
ax.set(aspect='equal', title='Analytic probability-flow trajectories')
plt.show()

In [ ]:
seed = np.array([1.25, -.55])
trajectory = np.array([flow_map(seed, 0, t) for t in ts])
exact_out = np.array([f_exact(x, t) for x, t in zip(trajectory, ts)])
surrogate_out = np.array([f_surrogate(x, t) for x, t in zip(trajectory, ts)])
fig, axes = plt.subplots(1, 2, figsize=(10, 3.5), constrained_layout=True)
axes[0].plot(ts, exact_out[:, 0], label='exact $f$, coordinate 1')
axes[0].plot(ts, exact_out[:, 1], label='exact $f$, coordinate 2')
axes[0].axhline(seed[0], color='0.4', ls=':')
axes[0].axhline(seed[1], color='0.4', ls=':')
axes[0].set(title='Exact output is constant on one trajectory', xlabel='t')
axes[0].legend(fontsize=8)
axes[1].plot(ts, np.linalg.norm(surrogate_out - seed, axis=1), color='tab:red')
axes[1].set(title='Injected surrogate consistency error', xlabel='t', ylabel='$||f_\theta-x_0||$')
plt.show()

In [ ]:
def cd_residual(f, x0, t, delta):
    s = max(0.0, t - delta)
    xt, xs = flow_map(x0, 0, t), flow_map(x0, 0, s)
    return np.linalg.norm(f(xt, t) - f(xs, s), axis=-1).mean()

def ct_residual(f, x0, t, h=1e-4):
    plus = f(flow_map(x0, 0, t + h), t + h)
    minus = f(flow_map(x0, 0, t - h), t - h)
    return np.linalg.norm((plus - minus) / (2 * h), axis=-1).mean()

batch = rng.normal(size=(500, 2))
deltas = np.logspace(-3, -0.35, 18)
cd_exact = np.array([cd_residual(f_exact, batch, .7, d) for d in deltas])
cd_surr = np.array([cd_residual(f_surrogate, batch, .7, d) for d in deltas])
print('CT residual, exact:', ct_residual(f_exact, batch, .7))
print('CT residual, surrogate:', ct_residual(f_surrogate, batch, .7))
fig, ax = plt.subplots(figsize=(6, 4))
ax.loglog(deltas, np.maximum(cd_exact, 1e-15), label='exact f')
ax.loglog(deltas, cd_surr, label='surrogate f')
ax.set(xlabel=r'$\Delta t$', ylabel='CD residual', title='Finite differences approach the CT condition')
ax.legend()
plt.show()

In [ ]:
def vector_field(x, t):
    x = np.asarray(x)
    Jx = np.stack([-x[..., 1], x[..., 0]], axis=-1)
    return -DECAY * x + 2 * OMEGA * t * Jx

def reverse_euler(x1, n_steps):
    x = np.array(x1, copy=True)
    dt = -1.0 / n_steps
    for k in range(n_steps):
        t = 1.0 + k * dt
        x = x + dt * vector_field(x, t)
    return x

clean = rng.normal(size=(800, 2))
noisy_end = flow_map(clean, 0, 1)
steps = np.array([1, 2, 4, 8, 16, 32, 64])
rmse = np.array([np.sqrt(np.mean((reverse_euler(noisy_end, n) - clean) ** 2)) for n in steps])
fig, ax = plt.subplots(figsize=(6, 4))
ax.loglog(steps, rmse, 'o-')
ax.set(xlabel='consistency / solver steps', ylabel='endpoint RMSE',
       title='One-step is the hardest compression point')
ax.grid(True, which='both', alpha=.25)
plt.show()

## 讀者練習 / TODO
在 `f_surrogate` 加上一個不依賴 $t$ 的常數偏移。它會讓 endpoint 變差，但 CD／CT residual 會不會發現？用實驗說明為什麼 consistency condition 還必須搭配邊界條件。